In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.6 MB/s eta 0:00:00


In [ ]:
import re, string, pickle, pandas as pd, numpy as np

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

**preproces**

In [ ]:
# ==========================
# LOAD DATA
# ==========================
df = pd.read_csv("data_labeled_manual.csv")

# ==========================
# STEMMER
# ==========================
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# ==========================
# STOPWORDS AMAN (tidak hapus kata penting)
# ==========================
stopwords_custom = set([
    "yang","untuk","dan","di","ke","dari","ini","itu","juga","kami","kamu","saya","meirizka",
    "ada","dengan","pada","karena","agar","atau","jadi","tidak","iya","yg","nya","ya","tapi","powwl","ny","nih","lahh","zxz",
    "buat","dalam","para","akan","sudah","serta","sih","hehe","hmm","deh","mbak","mba","pool","tania",
    "mas","kak","caroline","devita","lena","dg","fifin","tp","dgn","loh","t","b","aja","imo","amoun",
    "saja","pun","itu","anda","agar","yakni","sebagai","maka","yaitu","silvia","ad","sy","san","kpd","the","untvlantai","lo",
    "bs","d","n","sll","aztuti","dkt", "tx", "u", "anak","aku","devita","best","dj","thania","fres","asa",
    "dll","yak","hahaha","ken","siip","kita","madam","diksh","kecil","h","yahh","x","mb","st","fo","silviaa","ka",
    "f","b","sm","tdk","nur","si","kap","m","yuuk","parpel","chelsi","dg","sgt,","tpi","se","occ","daan",
    "an","andin","traine","trainee","s","mau","kes","asa","jd","ku","dik","welcome","dr","thania"
])

# KATA PENTING DOMAIN HOTEL — JANGAN DIHAPUS
stopword_protect = {"hotel","kamar","lokasi","pelayanan","fasilitas","staf","sarapan",
                    "bersih","nyaman","air","ac","wifi","parkir","resepsionis",
                    "checkin","checkout","kasur","bantal"}
stopwords = stopwords_custom - stopword_protect

# ==========================
# NORMALISASI SLANG (fix conflict & duplicates)
# ==========================
slang = {
    "bgtt":"banget","bgt":"banget","bangett":"banget","byk":"banyak","inn":"in","bnget":"banget","kren":"keren","suuperr":"bagus","servicenha":"servis",
    "gpp":"gapapa","ga":"tidak","gk":"tidak","nggak":"tidak","ngga":"tidak","kram":"kran","dkt":"dekat","cek":"check","pesen":"pesan",
    "makasih":"terima kasih","mksh":"terima kasih","thx":"terima kasih","jl":"jalan","dtg":"datang","baguss":"bagus","thank you":"terimakasih",
    "oke":"baik","ok":"baik","sip":"baik","bberapa":"beberapa","asa":"aja","lg":"lagi","msk":"masuk","tv":"televisi","clear":"bersih","tsb":"tersebut","overalla":"over all",
    "rekomen":"rekomendasi","recommended":"rekomendasi","sangatt":"sangat","maknyus":"enak","topp":"bagus","thank u":"terimaksih","tengkyu":"terimakasih",
    "jg":"juga","jd":"jadi","dpt":"dapat","dapet":"dapat","apalgii":"apalagi","restorannlt":"restoran","desertnya":"desert","trmmksh":"terimakasih",
    "km":"kamu","tmn":"teman","tmpt":"tempat","tks":"terimakasih","berasa":"rasa","ksni":"kesini","sus":"aneh","baguus":"bagus","besty":"best","trimakasih":"terimakasih",
    "udh":"sudah","sdh":"sudah","blm":"belum","hr":"hari","wlopun":"walaupun","happy":"senang","bussines trip":"perjalanan bisnis","ga":"tidak",
    "kl":"kalau","klo":"kalau","kalo":"kalau","tksh":"terimakasih","cantikk":"cantik","nggk":"tidak","menservice":"servis","overall":"semua","sat set":"cepat",
    "krn":"karena","trm":"terima","trs":"terus","bgs":"bagus","breakfastnya":"breakfast","enakk":"enak","thank":"terimakasih","lt":"lantai",
    "bbrp":"beberapa","kmr":"kamar","dgn":"dengan","lbh":"lebih","enakk":"enak","baguss":"bagus","trus":"terus","gak":"tidak","bnyk":"banyak","amburadul":"berantakan",
    "mantappp":"mantap","mantapppp":"mantap","mantappppp":"mantap","sukakk":"suka","lonsay":"lontong sayur","too bad":"buruk","deal":"setuju","tgl":"tanggal","agam":"agak","kg":"juga",
    "parahh":"parah","pdhal":"padahal","tpat":"tempat","lg":"lagi","yogjakarta":"yogyakarta","pelayannann":"pelayanan","recomend":"recomended",
    "thank u":"terima kasih","utk":"untuk","org":"orang","pengkap":"lengkap","rs":"rumah sakit","breakfast":"sarapan","pdhl":"padahal","bukber":"buka bersama","hotell":"hotel"
}

def normalize_slang(word):
    return slang.get(word, word)

# ==========================
# FULL PREPROCESSING FUNCTION
# ==========================
def preprocess_full(text):
    if pd.isna(text):
        return ""

    # 1. Casefolding
    t = text.lower()

    # 2. Clean HTML tags
    t = re.sub(r"<.*?>", " ", t)

    # 3. Remove URL
    t = re.sub(r"http\S+|www\.\S+", "", t)

    # 4. Remove @mention
    t = re.sub(r"@\w+", "", t)

    # 5. Remove emoji
    t = t.encode('ascii', 'ignore').decode('ascii')

    # 6. Remove punctuation
    t = t.translate(str.maketrans("", "", string.punctuation))

    # 7. Remove numbers
    t = re.sub(r"\d+", "", t)

    # 8. Tokenize words
    tokens = t.split()

    # 9. Normalisasi slang
    tokens = [normalize_slang(w) for w in tokens]

    # 10. Remove repeated characters
    tokens = [re.sub(r"(.)\1{2,}", r"\1", w) for w in tokens]

    # 11. Stopword removal
    tokens = [w for w in tokens if w not in stopwords]

    # 12. Stemming Sastrawi
    tokens = [stemmer.stem(w) for w in tokens]

    # 13. Remove extra whitespace
    return " ".join(tokens)

# ==========================
# APPLY TO DATAFRAME
# ==========================
df["processed"] = df["Casefold"].astype(str).apply(preprocess_full)

df.to_csv("data_labeled_manual_preprocessed.csv", index=False)

print("Preprocessing FINAL selesai! File: data_labeled_manual_preprocessed.csv")
df.head()

Preprocessing FINAL selesai! File: data_labeled_manual_preprocessed.csv


,Label,Casefold,processed
0,positif,pelayanannya ramah bgtt fasilitas kamar juga o...,layan ramah banget fasilitas kamar baik rekome...
1,negatif,baru kali ini sangat kecewa dengan pelayanan h...,baru kali sangat kecewa layan hotel mercure ko...
2,positif,liburan menyenangkan di jogja stay di hotel yg...,libur senang jogja stay hotel luar biasa
3,positif,lokasinya juga mantap dekat dengan bandara dan...,lokasi mantap dekat bandara tempat tempat wisa...
4,positif,saya merekomendasikan anda untuk menginap di h...,rekomendasi inap hotel layan bagus ramah makan...


**Memisah data yang sudah dilabel manual untuk Train**

In [ ]:
# Load dataset (sudah berlabel dan tidak)
dataset = pd.read_csv('/content/data_labeled_manual_preprocessed.csv')

manual = dataset[~dataset['Label'].isna()].copy() # Corrected column name and added .copy()
unlabeled = dataset[dataset['Label'].isna()].copy() # Corrected column name and added .copy()

manual['processed'] = manual['processed'].fillna("").astype(str)
unlabeled['processed'] = unlabeled['processed'].fillna("").astype(str)



**Buat Tokenizer**

In [ ]:
# Tokenizer
max_words = 10000   # membatasi kosa kata (10k berarti bisa banyak bgt)
max_len = 100       # membatasi banyaknya token per data

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(manual['processed'])

# Save tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [ ]:
def preprocess_text(df, data_tokenizer):
    df = df.copy()
    df['processed'] = df['processed'].fillna("").astype(str)

    sequences = data_tokenizer.texts_to_sequences(df['processed'])
    padded = pad_sequences(sequences, maxlen=max_len)
    return padded

**Train Model Awal**

In [ ]:
# Encode label
le = LabelEncoder()
manual['label_encoded'] = le.fit_transform(manual['Label'])

# Load tokenizer
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Preprocess
X_labeled = preprocess_text(manual, tokenizer)
y_labeled = manual['label_encoded'].values

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X_labeled, y_labeled, test_size=0.2, random_state=42)

# Build model
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128)) # Removed deprecated input_length argument
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(len(manual['Label'].unique()), activation='softmax')) # Corrected column name

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=15, batch_size=32, callbacks=[early_stop])

# Save model
model.save('model_initial.h5') # Consider changing to 'model_initial.keras' for the latest format
print("Initial model trained and saved as model_initial.h5")

Epoch 1/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 279ms/step - accuracy: 0.4331 - loss: 1.0876 - val_accuracy: 0.7200 - val_loss: 0.9983
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 227ms/step - accuracy: 0.6233 - loss: 1.0386 - val_accuracy: 0.6800 - val_loss: 0.8899
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 134ms/step - accuracy: 0.6214 - loss: 0.9653 - val_accuracy: 0.7200 - val_loss: 0.7344
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 132ms/step - accuracy: 0.6502 - loss: 0.8002 - val_accuracy: 0.8000 - val_loss: 0.6586
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step - accuracy: 0.6511 - loss: 0.7875 - val_accuracy: 0.8400 - val_loss: 0.6255
Epoch 6/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step - accuracy: 0.7623 - loss: 0.6149 - val_accuracy: 0.8600 - val_loss: 0.5309
Epoch 7/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step - accuracy: 0.7892 - loss: 0.4683 - val_accuracy: 0.8600 - val_loss: 0.4628
Epoch 8/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step - accuracy: 0.8038 - loss: 0.3748 - val_accuracy: 0.8800 - val_loss:

Initial model trained and saved as model_initial.h5


**Labeling dengan Model Awal untuk Self Train**

In [91]:
# Load model awal
model = load_model('model_initial.h5')

with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Preprocess unlabeled data
X_unlabeled = preprocess_text(unlabeled, tokenizer)

# Prediksi pseudo-label
pred_probs = model.predict(X_unlabeled)
pred_labels = np.argmax(pred_probs, axis=1)
pred_confidence = np.max(pred_probs, axis=1)

# Threshold confidence
threshold = 0.85
high_conf_idx = np.where(pred_confidence >= threshold)[0]

unlabeled = unlabeled.reset_index(drop=True)
manual = manual.reset_index(drop=True)

pseudo_data = unlabeled.iloc[high_conf_idx].copy()
pseudo_data['label_encoded'] = pred_labels[high_conf_idx]
pseudo_data['model_Label'] = le.inverse_transform(pseudo_data['label_encoded'])

# Gabungkan manual + pseudo-labeled
train_self = pd.concat([manual, pseudo_data], ignore_index=True)
train_self['model_Label'] = le.inverse_transform(train_self['label_encoded'])

# Simpan ke CSV
train_self.to_csv('data_labeled_for_self_train.csv', index=False)

31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step


**Self Train Model Awal dengan data yang dilabel oleh Model**

In [92]:
# Load self-train dataset
train_self = pd.read_csv('data_labeled_for_self_train.csv')

train_self['processed'] = train_self['processed'].astype(str)


In [93]:
# Load tokenizer
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Preprocess
X_self = preprocess_text(train_self, tokenizer)
y_self = train_self['label_encoded'].values

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X_self, y_self, test_size=0.2, random_state=42)

# Build model
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128)) # Removed deprecated input_length argument
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(len(train_self['label_encoded'].unique()), activation='softmax')) # Corrected column name to 'label_encoded'

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=10, batch_size=32, callbacks=[early_stop])

# Save final model
model.save('model_self_trained.h5')

Epoch 1/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 181ms/step - accuracy: 0.7302 - loss: 0.7807 - val_accuracy: 0.8357 - val_loss: 0.3615
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 134ms/step - accuracy: 0.8332 - loss: 0.3876 - val_accuracy: 0.9420 - val_loss: 0.2131
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 131ms/step - accuracy: 0.9122 - loss: 0.2250 - val_accuracy: 0.9517 - val_loss: 0.1370
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 134ms/step - accuracy: 0.8993 - loss: 0.1873 - val_accuracy: 0.9662 - val_loss: 0.1076
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 5s 174ms/step - accuracy: 0.9474 - loss: 0.1090 - val_accuracy: 0.9614 - val_loss: 0.1160
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 132ms/step - accuracy: 0.9774 - loss: 0.0686 - val_accuracy: 0.9565 - val_loss: 0.1136
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 133ms/step - accuracy: 0.9887 - loss: 0.0513 - val_accuracy: 0.9420 - val_loss: 0.1255


**labeling dengan LLM (ChatGPT)**

In [94]:
from google.colab import userdata

In [116]:
from openai import OpenAI
import json
import time

client = OpenAI(api_key=userdata.get('api_key_OpenAI'))

In [146]:
self_train_data = pd.read_csv("data_labeled_for_self_train.csv")
self_train_data.to_csv("data.csv", index=False)

df = pd.read_csv("data.csv")      # datasetnya diduplikat trs diganti aja namanya jd ini

In [147]:
if "gpt_label" not in df.columns:
    df["gpt_label"] = ""

In [148]:
df.drop("Label", axis=1, inplace=True)

In [150]:
unlabeled = df[df["gpt_label"].isna() | (df["gpt_label"] == "")]
batch_size = 20               # ini biar sekali ngeprompt ngambil banyak data dan dapat sentimennya sebanyak data itu. biar ga 1 data 1 prompt, keberatan, boros saldo.
model_gpt = "gpt-5-mini"

topik = "hotel grand mercure"

In [152]:
for start in range(0, len(unlabeled), batch_size):
    batch = unlabeled.iloc[start:start+batch_size]

    texts = batch["Casefold"].tolist()
    indexed_texts = "\n".join([f"{i+1}. {t}" for i, t in enumerate(texts)])

    prompt = f"""
    Kamu adalah model analisis sentimen. Topik pembahasan: "{topik}"

    Tugasmu: Tentukan sentimen untuk SETIAP teks (positif, netral, atau negatif).

    Kriteria keputusan:
    - "positif": jika teks mayoritas berisi pujian, kepuasan, dukungan, atau nada setuju.
    - "negatif": jika teks mayoritas berisi keluhan, ketidakpuasan, kritik kuat, atau nada tidak setuju.
    - "netral": jika teks tidak condong ke positif maupun negatif, termasuk teks yang hanya memberikan informasi, saran ringan, atau pertanyaan.

    Aturan konflik:
    - Jika dalam satu kalimat terdapat pujian dan kritik, tentukan sentimen berdasarkan bagian yang **lebih dominan atau lebih kuat**.
      - Kritik kuat + pujian kecil → negatif.
      - Pujian kuat + keluhan kecil → positif.
      - Jika keduanya ringan dan seimbang → netral.

    Format jawaban:
    Balas dalam FORMAT JSON PLAIN tanpa teks tambahan dan tanpa ```.

    Contoh:
    {{"1": "positif", "2": "negatif", "3": "netral", "4": "positif", "5": "positif"}}

    Teks yang harus dianalisis:
    {indexed_texts}
    """
    # print(indexed_texts)

    try:
        response = client.chat.completions.create(
            model= model_gpt,
            messages=[{"role": "user", "content": prompt}],
        )

        result_text = response.choices[0].message.content.strip()

        #  Bersihin kalau model nambahin ```json ... ```
        result_text = re.sub(r"^```(?:json)?|```$", "", result_text, flags=re.MULTILINE).strip()

        try:
            sentiments = json.loads(result_text)
        except json.JSONDecodeError:
            print(" Gagal parse JSON, hasil mentah:", result_text)
            continue

        for i, (_, row) in enumerate(batch.iterrows()):
            df.at[row.name, "gpt_label"] = sentiments.get(str(i+1), "")

        df.to_csv("data.csv", index=False)
        print(f"✅ Batch {start}-{start+batch_size} selesai")

        time.sleep(2)

    except Exception as e:
        err = str(e)
        if "429" in err:
            print(" Kena rate limit, tidur dulu 25 detik...")
            time.sleep(25)
            continue
        else:
            print(f" Error batch {start}: {e}")
            time.sleep(5)

✅ Batch 0-20 selesai
✅ Batch 20-40 selesai
✅ Batch 40-60 selesai
✅ Batch 60-80 selesai
✅ Batch 80-100 selesai
✅ Batch 100-120 selesai
✅ Batch 120-140 selesai
✅ Batch 140-160 selesai
✅ Batch 160-180 selesai
✅ Batch 180-200 selesai
✅ Batch 200-220 selesai
✅ Batch 220-240 selesai
✅ Batch 240-260 selesai
✅ Batch 260-280 selesai
✅ Batch 280-300 selesai
✅ Batch 300-320 selesai
✅ Batch 320-340 selesai
✅ Batch 340-360 selesai
✅ Batch 360-380 selesai
✅ Batch 380-400 selesai
✅ Batch 400-420 selesai
✅ Batch 420-440 selesai
✅ Batch 440-460 selesai
✅ Batch 460-480 selesai
✅ Batch 480-500 selesai
✅ Batch 500-520 selesai
✅ Batch 520-540 selesai
✅ Batch 540-560 selesai
✅ Batch 560-580 selesai
✅ Batch 580-600 selesai
✅ Batch 600-620 selesai
✅ Batch 620-640 selesai
✅ Batch 640-660 selesai
✅ Batch 660-680 selesai
✅ Batch 680-700 selesai
✅ Batch 700-720 selesai
✅ Batch 720-740 selesai
✅ Batch 740-760 selesai
✅ Batch 760-780 selesai
✅ Batch 780-800 selesai
✅ Batch 800-820 selesai
✅ Batch 820-840 selesai
✅ 

In [153]:
positif_count = (df["gpt_label"] == "positif").sum()
negatif_count = (df["gpt_label"] == "negatif").sum()

print("Jumlah positif:", positif_count)
print("Jumlah negatif:", negatif_count)

Jumlah positif: 909
Jumlah negatif: 92
